## Packages and Codes

In [ ]:
%run -i ./00.0_setup_workspace_env.py

try:
    import polars as pl
except ImportError:  # install only if the image does not already have it
    get_ipython().run_line_magic("pip", "install -q polars")
    import polars as pl

from IPython.display import display
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

nlp_ds = f"{WORKSPACE_CDR.split('.')[0]}.C_V9_R6_offcycle_nlp"

In [ ]:
# This line allows for the plots to be displayed inline in the Jupyter notebook
# %matplotlib inline

# # sns.set(style="whitegrid",font_scale=0.9)

palette = ['#0173b2', '#de8f05', '#8de5a1', '#d55e00', '#029e73', '#cc78bc', '#ece133', 
           '#56b4e9', '#ca9161', '#fbafe4', '#949494']

sns.set_palette(sns.color_palette(palette))
sns.color_palette(palette)

In [ ]:
# show all columns in pandas
pd.set_option("display.max_columns", None)

# show full column width
pd.set_option('display.max_colwidth', 100)

# Polars string length to 100
pl.Config.set_fmt_str_lengths(100)

pl.Config.set_tbl_rows(50)

## Helper Functions

In [ ]:
def polars_gbq(query):
    """
    Take a SQL query and return result as polars dataframe
    :param query: BigQuery SQL query
    :return: polars dataframe
    """
    client = bigquery.Client()
    query_job = client.query(query)
    rows = query_job.result()
    df = pl.from_arrow(rows.to_arrow())

    return df

def percentile_dates_from_monthly_counts(df, column, qs=(0.25, 0.50, 0.75)):
    """
    Find dates where cumulative counts reach 25%, 50%, and 75% of the total count.
    """
    values = df[column].fillna(0).to_numpy(dtype=float)
    cumulative = np.cumsum(values)
    total = cumulative[-1]

    if total == 0:
        return {}

    percentile_dates = {}
    for q in qs:
        idx = np.searchsorted(cumulative, q * total)
        idx = min(idx, len(df) - 1)
        percentile_dates[q] = pd.to_datetime(df.index[idx])

    return percentile_dates

def round_up_to_nearest(value, base=100000):
    """
    Round value up to the nearest `base`.
    """
    if value <= 0:
        return base
    return int(np.ceil(value / base) * base)

## EHR Data Density

Types of PPI: Observation, Meas Value, Measurement, Condition, Drug  
Survey concepts in observation: concept_id = "PPI" AND domain_id = "Condition"

### Create DFs by month

In [ ]:
# surveys per month for whole cohort
survey_months_q = f"""
WITH surveys AS (
    SELECT person_id AS person, o.observation_id AS survey_id, o.observation_date AS date
    FROM {WORKSPACE_CDR}.observation AS o
    JOIN {WORKSPACE_CDR}.concept AS c ON o.observation_source_value = c.concept_code
    WHERE c.vocabulary_id = 'PPI' AND c.domain_id = 'Observation'
)
SELECT
    DATE_TRUNC(date, MONTH) AS month,
    COUNT(DISTINCT survey_id) AS survey_count,
    COUNT(DISTINCT person) AS survey_person_count
FROM surveys
GROUP BY month
ORDER BY month
"""

survey_months_df = polars_gbq(survey_months_q)

In [ ]:
# procedures per month for whole cohort
procedures_months_q = f"""
WITH procedures AS (
    SELECT CONCAT('po-', CAST(p.procedure_occurrence_id AS STRING)) AS procedure_id,
           p.person_id AS person,
           p.procedure_date AS date
    FROM {WORKSPACE_CDR}.procedure_occurrence AS p
    JOIN {WORKSPACE_CDR}.concept AS c ON p.procedure_concept_id = c.concept_id
    WHERE c.domain_id = 'Procedure'

    UNION ALL

    SELECT CONCAT('obs-', CAST(o.observation_id AS STRING)),
           o.person_id,
           o.observation_date
    FROM {WORKSPACE_CDR}.observation AS o
    JOIN {WORKSPACE_CDR}.concept AS c ON o.observation_source_concept_id = c.concept_id
    WHERE c.vocabulary_id IN ('CPT4', 'HCPCS', 'ICD9Proc', 'ICD10PCS')

    UNION ALL

    SELECT CONCAT('meas-', CAST(m.measurement_id AS STRING)),
           m.person_id,
           m.measurement_date
    FROM {WORKSPACE_CDR}.measurement AS m
    JOIN {WORKSPACE_CDR}.concept AS c ON m.measurement_source_concept_id = c.concept_id
    WHERE c.vocabulary_id IN ('CPT4', 'HCPCS')
)
SELECT
    DATE_TRUNC(date, MONTH) AS month,
    COUNT(DISTINCT procedure_id) AS procedure_count,
    COUNT(DISTINCT person) AS procedure_person_count
FROM procedures
GROUP BY month
ORDER BY month
"""

procedures_months_df = polars_gbq(procedures_months_q)

In [ ]:
# conditions per month for whole cohort
conditions_months_q = f"""
WITH conditions AS (
    SELECT person_id AS person,
           CONCAT('obs-', CAST(o.observation_id AS STRING)) AS condition_id,
           o.observation_date AS date
    FROM {WORKSPACE_CDR}.observation AS o
    JOIN {WORKSPACE_CDR}.concept AS c ON o.observation_source_value = c.concept_code
    WHERE c.vocabulary_id IN ('ICD9CM', 'ICD10CM', 'SNOMED')

    UNION ALL

    SELECT person_id AS person,
           CONCAT('obs-', CAST(o.observation_id AS STRING)) AS condition_id,
           o.observation_date AS date
    FROM {WORKSPACE_CDR}.observation AS o
    JOIN {WORKSPACE_CDR}.concept AS c ON o.observation_source_concept_id = c.concept_id
    WHERE c.vocabulary_id IN ('ICD9CM', 'ICD10CM', 'SNOMED')

    UNION ALL

    SELECT person_id AS person,
           CONCAT('co-', CAST(co.condition_occurrence_id AS STRING)) AS condition_id,
           co.condition_start_date AS date
    FROM {WORKSPACE_CDR}.condition_occurrence AS co
    JOIN {WORKSPACE_CDR}.concept AS c ON co.condition_source_value = c.concept_code
    WHERE c.vocabulary_id IN ('ICD9CM', 'ICD10CM', 'SNOMED')

    UNION ALL

    SELECT person_id AS person,
           CONCAT('co-', CAST(co.condition_occurrence_id AS STRING)) AS condition_id,
           co.condition_start_date AS date
    FROM {WORKSPACE_CDR}.condition_occurrence AS co
    JOIN {WORKSPACE_CDR}.concept AS c ON co.condition_source_concept_id = c.concept_id
    WHERE c.vocabulary_id IN ('ICD9CM', 'ICD10CM', 'SNOMED')
)
SELECT
    DATE_TRUNC(date, MONTH) AS month,
    COUNT(DISTINCT condition_id) AS condition_count,
    COUNT(DISTINCT person) AS condition_person_count
FROM conditions
GROUP BY month
ORDER BY month
"""

conditions_months_df = polars_gbq(conditions_months_q)

In [ ]:
# measurements per month for whole cohort
measurements_months_q = f"""
WITH measurements AS (
    SELECT
        m.person_id AS person,
        m.measurement_id,
        m.measurement_date AS date,
        CASE
            WHEN LOWER(TRIM(CAST(me.src_id AS STRING))) LIKE '%ehr%'
                THEN 'ehr'

            WHEN TRIM(CAST(me.src_id AS STRING)) IN (
                'Participant Portal: TPC',
                'Participant Portal: PTSC',
                'Staff Portal: HealthPro'
            )
                THEN 'ppi'

            ELSE 'unknown'
        END AS source_type

    FROM {WORKSPACE_CDR}.measurement AS m
    JOIN {WORKSPACE_CDR}.concept AS c
        ON m.measurement_concept_id = c.concept_id
    LEFT JOIN {WORKSPACE_CDR}.measurement_ext AS me
        ON m.measurement_id = me.measurement_id

    WHERE c.domain_id = 'Measurement'
),

monthly_counts AS (
    SELECT
        DATE_TRUNC(date, MONTH) AS month,

        COUNT(DISTINCT IF(source_type = 'ehr', measurement_id, NULL))
            AS ehr_measurement_count,

        COUNT(DISTINCT IF(source_type = 'ehr', person, NULL))
            AS ehr_measurement_person_count,

        COUNT(DISTINCT IF(source_type = 'ppi', measurement_id, NULL))
            AS ppi_measurement_count,

        COUNT(DISTINCT IF(source_type = 'ppi', person, NULL))
            AS ppi_measurement_person_count,

        COUNT(DISTINCT IF(source_type = 'unknown', measurement_id, NULL))
            AS unknown_measurement_count,

        COUNT(DISTINCT IF(source_type = 'unknown', person, NULL))
            AS unknown_measurement_person_count

    FROM measurements
    GROUP BY month
)

SELECT *
FROM monthly_counts
ORDER BY month
"""

measurements_months_df = polars_gbq(measurements_months_q)

In [ ]:
# drugs per month for whole cohort
drugs_months_q = f"""
WITH drugs AS (
    SELECT person_id AS person, de.drug_exposure_id AS drug_id, de.drug_exposure_start_date AS date
    FROM {WORKSPACE_CDR}.drug_exposure AS de
    JOIN {WORKSPACE_CDR}.concept AS c ON de.drug_concept_id = c.concept_id
    WHERE c.domain_id = 'Drug'
)
SELECT
    DATE_TRUNC(date, MONTH) AS month,
    COUNT(DISTINCT drug_id) AS drug_count,
    COUNT(DISTINCT person) AS drug_person_count
FROM drugs
GROUP BY month
ORDER BY month
"""

drugs_months_df = polars_gbq(drugs_months_q)

In [ ]:
# wearables per month for whole cohort
wearables_months_q = f"""
WITH wearables AS (
    SELECT
        person_id AS person,
        CONCAT(CAST(person_id AS STRING), '-', FORMAT_DATE('%F', act.date)) AS wearable_id,
        act.date AS date
    FROM {WORKSPACE_CDR}.activity_summary AS act

    UNION ALL

    SELECT
        person_id AS person,
        CONCAT(CAST(person_id AS STRING), '-', FORMAT_DATE('%F', sds.sleep_date)) AS wearable_id,
        sds.sleep_date AS date
    FROM {WORKSPACE_CDR}.sleep_daily_summary AS sds

    UNION ALL

    SELECT
        person_id AS person,
        CONCAT(CAST(person_id AS STRING), '-', FORMAT_DATE('%F', hrs.date)) AS wearable_id,
        hrs.date AS date
    FROM {WORKSPACE_CDR}.heart_rate_summary AS hrs
)
SELECT
    DATE_TRUNC(date, MONTH) AS month,
    COUNT(DISTINCT wearable_id) AS wearable_count,
    COUNT(DISTINCT person) AS wearable_person_count
FROM wearables
GROUP BY month
ORDER BY month
"""
wearables_months_df = polars_gbq(wearables_months_q)

In [ ]:
# notes per month for whole cohort
## for nlp table, use "note_nlp" instead of "note" table
notes_months_q = f"""
WITH notes AS (
    SELECT
        nlp.note_id   AS note_id,
        nlp.person_id AS person,
        nlp.note_date AS date
    FROM {nlp_ds}.note AS nlp
)
SELECT
    DATE_TRUNC(date, MONTH) AS month,
    COUNT(DISTINCT note_id) AS note_count,
    COUNT(DISTINCT person)  AS note_person_count
FROM notes
GROUP BY month
ORDER BY month
"""
notes_months_df = polars_gbq(notes_months_q)

In [ ]:
# cognitive tasks per month for whole cohort
cognitive_months_q = f"""
WITH cognitive_tasks AS (
    SELECT
        person_id AS person,
        CONCAT(CAST(person_id AS STRING), '-', FORMAT_DATE('%F', DATE(dd.metadata.test_start_date_time))) AS task_id,
        DATE(dd.metadata.test_start_date_time) AS date
    FROM {WORKSPACE_CDR}.delaydiscounting AS dd

    UNION ALL

    SELECT
        person_id AS person,
        CONCAT(CAST(person_id AS STRING), '-', FORMAT_DATE('%F', DATE(er.metadata.test_start_date_time))) AS task_id,
        DATE(er.metadata.test_start_date_time) AS date
    FROM {WORKSPACE_CDR}.emorecog AS er

    UNION ALL

    SELECT
        person_id AS person,
        CONCAT(CAST(person_id AS STRING), '-', FORMAT_DATE('%F', DATE(gc.metadata.test_start_date_time))) AS task_id,
        DATE(gc.metadata.test_start_date_time) AS date
    FROM {WORKSPACE_CDR}.gradcpt AS gc

    UNION ALL

    SELECT
        person_id AS person,
        CONCAT(CAST(person_id AS STRING), '-', FORMAT_DATE('%F', DATE(fl.metadata.test_start_date_time))) AS task_id,
        DATE(fl.metadata.test_start_date_time) AS date
    FROM {WORKSPACE_CDR}.flanker AS fl
)
SELECT
    DATE_TRUNC(date, MONTH) AS month,
    COUNT(DISTINCT task_id) AS task_count,
    COUNT(DISTINCT person) AS task_person_count
FROM cognitive_tasks
GROUP BY month
ORDER BY month
"""
cognitive_months_df = polars_gbq(cognitive_months_q)

In [ ]:
merged_months_df = (
    procedures_months_df
    .join(survey_months_df, on="month", how="full", coalesce=True)
    .join(conditions_months_df, on="month", how="full", coalesce=True)
    .join(measurements_months_df, on="month", how="full", coalesce=True)
    .join(drugs_months_df, on="month", how="full", coalesce=True)
    .join(wearables_months_df, on="month", how="full", coalesce=True)
    .join(notes_months_df, on="month", how="full", coalesce=True)
    .join(cognitive_months_df, on="month", how="full", coalesce=True)
).sort("month")

## Plot

In [ ]:
# ##read in file if previously saved:
# merged_months_df = pl.read_csv(f'{WORKSPACE_DIR}/data/aou_v9_cohort.csv')

In [ ]:
def make_data_availability_fig(figsize, title_size, ytick_size, xtick_size, ylabel_size):
    """
    Build the full data-availability figure at a given size.
    Returns (fig, axes) so the caller can save it.
    """
    fig, axes = plt.subplots(
        len(columns_to_plot),
        1,
        figsize=figsize,
        sharex=True,
        dpi=300
    )
    if len(columns_to_plot) == 1:
        axes = [axes]

    for i, column in enumerate(columns_to_plot):
        ax = axes[i]
        sns.despine(ax=ax, top=True, left=True, right=False)

        x = df_plot.index
        y = df_plot[column].fillna(0)

        ax.plot(x, y, color=color_dict[column], linewidth=1.8)
        ax.fill_between(x, y, color=color_dict[column], alpha=0.22)

        percentile_dates = percentile_dates_from_monthly_counts(
            df_raw, column, qs=(0.25, 0.50, 0.75)
        )
        for d in percentile_dates.values():
            ax.axvline(d, linestyle="--", linewidth=1, color="#8a8a8a", alpha=0.8)

        total_n = int(df_raw[column].sum())
        ax.text(
            0.012, 0.82,
            f"{pretty_names[column]} (n = {total_n:,})",
            transform=ax.transAxes,
            fontsize=title_size,
            ha="left", va="top"
        )

        ymax_raw = np.nanmax(y)
        ymax = ymax_overrides.get(column, round_up_to_nearest(ymax_raw, 100000))
        ax.set_ylim(0, ymax)
        ax.set_yticks([0, ymax / 2, ymax])
        ax.set_yticklabels(["0", f"{ymax/2:,.0f}", f"{ymax:,.0f}"])
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position("right")
        ax.tick_params(
            axis='y', right=True, left=False,
            labelright=True, labelleft=False, labelsize=ytick_size
        )
        ax.tick_params(axis='x', labelsize=xtick_size)
        ax.spines["left"].set_visible(False)
        ax.grid(False)
        ax.set_ylabel("")

    axes[-1].set_xlim(pd.to_datetime("1980-01-01"), pd.to_datetime("2025-01-01"))
    axes[-1].xaxis.set_major_locator(mdates.YearLocator(base=5))
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    axes[-1].tick_params(axis='x', pad=10)

    fig.text(
        0.985, 0.5, "Record counts per month",
        rotation=-90, va="center", ha="center", fontsize=ylabel_size
    )

    plt.tight_layout(rect=(0, 0, 0.97, 1))
    return fig, axes

In [ ]:
color_dict = {
    'ehr_measurement_count': '#919A15',
    'condition_count': '#3D8A2E',
    'drug_count': '#028E99',
    'procedure_count': '#0168A9',
    'note_count': '#9E4589',
    'wearable_count': '#C23537',
    'survey_count': '#E96301',
    'ppi_measurement_count': '#C59527',
    'task_count': '#6A4638',
}
pretty_names = {
    'survey_count': 'Surveys',
    'task_count': 'Cognitive Tasks',
    'procedure_count': 'EHR Procedures',
    'condition_count': 'EHR Conditions',
    'ppi_measurement_count': 'Enrollment Physical Measurements',
    'ehr_measurement_count': 'EHR Labs and Measurements',
    'drug_count': 'EHR Drug Exposures',
    'note_count': 'EHR Clinical Note NLP Concepts',
    'wearable_count': 'Wearables'
}
columns_to_plot = [
    'ehr_measurement_count', 'condition_count', 'drug_count', 'procedure_count',
    'note_count', 'wearable_count', 'survey_count', 'ppi_measurement_count', 'task_count'
]

ymax_overrides = {
    'task_count': 15000,
    'note_count': 180000,
}

rolling_cols = [col for col in columns_to_plot if col in merged_months_df.columns]

df_raw = (
    merged_months_df.sort("month").fill_null(0).to_pandas().set_index("month")
)
df_wide = (
    merged_months_df.sort("month").fill_null(0)
    .with_columns([pl.col(col).rolling_mean(window_size=5).alias(col) for col in rolling_cols])
)
df_plot = df_wide.to_pandas().set_index("month")
columns_to_plot = [c for c in columns_to_plot if c in df_plot.columns]

In [ ]:
# ---- Version 1: publication sizing ----
fig_presentation, axes_presentation = make_data_availability_fig(
    figsize=(16, 1.65 * len(columns_to_plot)),
    title_size=27,
    ytick_size=22,
    xtick_size=22,
    ylabel_size=24,
)
plt.show()

os.makedirs(f'{WORKSPACE_DIR}/data', exist_ok=True)
fig_presentation.savefig(
    f"{WORKSPACE_DIR}/data/fig_1_aou_data_v9.pdf",
    dpi=300, bbox_inches="tight"
)

# ---- Version 2: 12" x 5.7" for ppt presentations, fonts scaled down to fit the shorter height ----
# Original used ~1.65"/panel at font sizes 22-27. Target height is 5.7" for 9
# panels (~0.63"/panel), roughly 38% of the original per-panel height, so font
# sizes are scaled down proportionally to avoid overlap.
scale = 5.7 / (1.65 * len(columns_to_plot))

fig_compact, axes_compact = make_data_availability_fig(
    figsize=(12, 5.7),
    title_size=max(6, round(27 * scale)),
    ytick_size=max(5, round(22 * scale)),
    xtick_size=max(5, round(22 * scale)),
    ylabel_size=max(6, round(24 * scale)),
)
plt.show()

fig_compact.savefig(
    f"{WORKSPACE_DIR}/data/fig_1_aou_data_v9_10x6.pdf",
    dpi=300, bbox_inches="tight"
)

In [ ]:
#save to avoid querying every time
os.makedirs(f'{WORKSPACE_DIR}/data', exist_ok=True)
merged_months_df.write_csv(f'{WORKSPACE_DIR}/data/aou_v9_cohort.csv')

In [ ]:
rows = []

for col in columns_to_plot:
    if col not in df_raw.columns:
        continue

    y = df_raw[col].fillna(0)

    if y.sum() == 0:
        continue

    percentile_dates = percentile_dates_from_monthly_counts(
        df_raw,
        col,
        qs=(0.25, 0.50, 0.75)
    )

    date_min = df_raw.index[y > 0].min().strftime("%Y-%m")
    date_max = df_raw.index[y > 0].max().strftime("%Y-%m")

    rows.append({
        "Data Type":  pretty_names[col],
        "Start":      date_min,
        "End":        date_max,
        "Q1 (25%)":   percentile_dates[0.25].strftime("%Y-%m"),
        "Median":     percentile_dates[0.50].strftime("%Y-%m"),
        "Q3 (75%)":   percentile_dates[0.75].strftime("%Y-%m"),
    })

summary_df = pd.DataFrame(rows)

In [ ]:
unique_persons_surveys_q = f"""
SELECT COUNT(DISTINCT o.person_id) AS unique_person_count
FROM {WORKSPACE_CDR}.observation AS o
JOIN {WORKSPACE_CDR}.concept AS c ON o.observation_source_value = c.concept_code
WHERE c.vocabulary_id = 'PPI' AND c.domain_id = 'Observation'
"""

unique_persons_procedures_q = f"""
SELECT COUNT(DISTINCT p.person_id) AS unique_person_count
FROM {WORKSPACE_CDR}.procedure_occurrence AS p
JOIN {WORKSPACE_CDR}.concept AS c ON p.procedure_concept_id = c.concept_id
WHERE c.domain_id = 'Procedure'
"""

unique_persons_conditions_q = f"""
SELECT COUNT(DISTINCT person_id) AS unique_person_count
FROM (
    SELECT person_id FROM {WORKSPACE_CDR}.observation AS o
    JOIN {WORKSPACE_CDR}.concept AS c ON o.observation_source_value = c.concept_code
    WHERE c.vocabulary_id IN ('ICD9CM', 'ICD10CM', 'SNOMED')
    UNION ALL
    SELECT person_id FROM {WORKSPACE_CDR}.observation AS o
    JOIN {WORKSPACE_CDR}.concept AS c ON o.observation_source_concept_id = c.concept_id
    WHERE c.vocabulary_id IN ('ICD9CM', 'ICD10CM', 'SNOMED')
    UNION ALL
    SELECT person_id FROM {WORKSPACE_CDR}.condition_occurrence AS co
    JOIN {WORKSPACE_CDR}.concept AS c ON co.condition_source_value = c.concept_code
    WHERE c.vocabulary_id IN ('ICD9CM', 'ICD10CM', 'SNOMED')
    UNION ALL
    SELECT person_id FROM {WORKSPACE_CDR}.condition_occurrence AS co
    JOIN {WORKSPACE_CDR}.concept AS c ON co.condition_source_concept_id = c.concept_id
    WHERE c.vocabulary_id IN ('ICD9CM', 'ICD10CM', 'SNOMED')
)
"""

unique_persons_measurements_ehr_q = f"""
SELECT
    COUNT(DISTINCT IF(LOWER(TRIM(CAST(me.src_id AS STRING))) LIKE '%ehr%', m.person_id, NULL)) AS ehr_unique_person_count,
    COUNT(DISTINCT IF(TRIM(CAST(me.src_id AS STRING)) IN (
        'Participant Portal: TPC',
        'Participant Portal: PTSC',
        'Staff Portal: HealthPro'
    ), m.person_id, NULL)) AS ppi_unique_person_count
FROM {WORKSPACE_CDR}.measurement AS m
JOIN {WORKSPACE_CDR}.concept AS c ON m.measurement_concept_id = c.concept_id
LEFT JOIN {WORKSPACE_CDR}.measurement_ext AS me ON m.measurement_id = me.measurement_id
WHERE c.domain_id = 'Measurement'
"""

unique_persons_drugs_q = f"""
SELECT COUNT(DISTINCT de.person_id) AS unique_person_count
FROM {WORKSPACE_CDR}.drug_exposure AS de
JOIN {WORKSPACE_CDR}.concept AS c ON de.drug_concept_id = c.concept_id
WHERE c.domain_id = 'Drug'
"""

unique_persons_wearables_q = f"""
SELECT COUNT(DISTINCT person_id) AS unique_person_count
FROM (
    SELECT person_id FROM {WORKSPACE_CDR}.activity_summary
    UNION ALL
    SELECT person_id FROM {WORKSPACE_CDR}.sleep_daily_summary
    UNION ALL
    SELECT person_id FROM {WORKSPACE_CDR}.heart_rate_summary
)
"""

unique_persons_cognitive_tasks_q = f"""
SELECT COUNT(DISTINCT person_id) AS unique_person_count
FROM (
    SELECT person_id FROM {WORKSPACE_CDR}.delaydiscounting
    UNION ALL
    SELECT person_id FROM {WORKSPACE_CDR}.emorecog
    UNION ALL
    SELECT person_id FROM {WORKSPACE_CDR}.gradcpt
    UNION ALL
    SELECT person_id FROM {WORKSPACE_CDR}.flanker
)
"""

unique_persons_notes_q = f"""
SELECT COUNT(DISTINCT note.person_id) AS unique_person_count
FROM {nlp_ds}.note
"""

# ehr_any definition: person has >=1 record across condition_occurrence,
# procedure_occurrence, drug_exposure, measurement, or observation whose
# src_id (via the matching _ext table) contains 'ehr' (case-insensitive).
unique_persons_ehr_q = f"""
WITH ext_union AS (
  SELECT t.person_id, ext.src_id FROM {WORKSPACE_CDR}.condition_occurrence t
    JOIN {WORKSPACE_CDR}.condition_occurrence_ext ext USING (condition_occurrence_id)
  UNION ALL
  SELECT t.person_id, ext.src_id FROM {WORKSPACE_CDR}.procedure_occurrence t
    JOIN {WORKSPACE_CDR}.procedure_occurrence_ext ext USING (procedure_occurrence_id)
  UNION ALL
  SELECT t.person_id, ext.src_id FROM {WORKSPACE_CDR}.drug_exposure t
    JOIN {WORKSPACE_CDR}.drug_exposure_ext ext USING (drug_exposure_id)
  UNION ALL
  SELECT t.person_id, ext.src_id FROM {WORKSPACE_CDR}.measurement t
    JOIN {WORKSPACE_CDR}.measurement_ext ext USING (measurement_id)
  UNION ALL
  SELECT t.person_id, ext.src_id FROM {WORKSPACE_CDR}.observation t
    JOIN {WORKSPACE_CDR}.observation_ext ext USING (observation_id)
)
SELECT COUNT(DISTINCT person_id) AS unique_person_count
FROM ext_union
WHERE LOWER(TRIM(CAST(src_id AS STRING))) LIKE '%ehr%'
"""

# Run all queries
results = {
    "surveys":            polars_gbq(unique_persons_surveys_q)["unique_person_count"][0],
    "procedures":         polars_gbq(unique_persons_procedures_q)["unique_person_count"][0],
    "conditions":         polars_gbq(unique_persons_conditions_q)["unique_person_count"][0],
    "ehr_measurements":   polars_gbq(unique_persons_measurements_ehr_q)["ehr_unique_person_count"][0],
    "ppi_measurements":   polars_gbq(unique_persons_measurements_ehr_q)["ppi_unique_person_count"][0],
    "drugs":              polars_gbq(unique_persons_drugs_q)["unique_person_count"][0],
    "wearables":          polars_gbq(unique_persons_wearables_q)["unique_person_count"][0],
    "cognitive_tasks":    polars_gbq(unique_persons_cognitive_tasks_q)["unique_person_count"][0],
    "notes":              polars_gbq(unique_persons_notes_q)["unique_person_count"][0],
    "ehr_any":            polars_gbq(unique_persons_ehr_q)["unique_person_count"][0],
}

unique_persons_df = pl.DataFrame({
    "data_type":          list(results.keys()),
    "unique_person_count": list(results.values()),
})

display(unique_persons_df)

In [ ]:
# % of records dated before 2010, per data type
print((df_raw.loc[df_raw.index < "2010-01-01", columns_to_plot].sum() / df_raw[columns_to_plot].sum() * 100).round(1))

In [ ]:
%run -i ./00.1_run_summary.py